# GV Tools — WFF PIERS0042 — 2026-08-10

Edit `INPUT_DIR` and `OUTPUT_DIR` in the next cell, then run all cells. Processing uses regular Python values and is safe inside Jupyter.

In [ ]:
from pathlib import Path
import os
import sys

GV_HOME = Path(
    os.environ.get("GV_TOOLS_HOME", Path.home() / "Desktop" / "Work" / "GV Tools")
).expanduser().resolve()
PACKAGE = GV_HOME / "gv_tools"
SOURCE = PACKAGE / "src"
if SOURCE.is_dir() and str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
if not PACKAGE.is_dir():
    raise FileNotFoundError(f"GV Tools project not found at {PACKAGE}. Set GV_TOOLS_HOME.")

DATE = "2026-08-10"
INPUT_DIR = Path(os.environ.get("GV_TOOLS_PIERS_INPUT", "/path/to/PIERS0042/Parsivel/input")).expanduser()
FILES = [
    INPUT_DIR / name
    for name in os.environ.get(
        "GV_TOOLS_PIERS_FILES", "PIERS0042_Parsivel_20260810_daily.zip"
    ).split(",")
    if name
]
OUTPUT_DIR = Path(os.environ.get("GV_TOOLS_OUTPUT", GV_HOME / "Output")).expanduser()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATE, INPUT_DIR, FILES, OUTPUT_DIR


In [ ]:
import gv_tools

if gv_tools.__version__ != "0.29.4":
    raise RuntimeError(
        f"This notebook requires GV Tools 0.29.4, but the Jupyter kernel has "
        f"{gv_tools.__version__} loaded. Restart the kernel, then Run All cells."
    )

In [ ]:
available_dates = gv_tools.io.discover_parsivel(FILES)
print("Available dates:", available_dates)
parsivel = gv_tools.io.read_parsivel(FILES)
parsivel


In [ ]:
created = gv_tools.io.write_product(parsivel, OUTPUT_DIR, formats=('netcdf', 'csv'), day=DATE, engine='h5netcdf')
created

In [ ]:
import matplotlib.pyplot as plt

figure_dir = gv_tools.graph.plot_directory(OUTPUT_DIR, 'Parsivel_Parameters', DATE)
figure_path = figure_dir / f'{SITE}_{INSTRUMENT_ID}_{YEAR:04d}{MONTH:02d}{DAY:02d}_parameters.png'

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True, constrained_layout=True)
ds.precipitation_rate.plot(ax=axes[0], color='tab:blue')
axes[0].set_title(f'{SITE} {INSTRUMENT_ID} Parsivel — {DATE}')
axes[0].set_ylabel('Rain rate (mm h$^{-1}$)')
ds.radar_reflectivity.plot(ax=axes[1], color='tab:orange')
axes[1].set_ylabel('Reflectivity (dBZ)')
fig.savefig(figure_path, dpi=200)
plt.show()
figure_path

In [ ]:
parsivel[['precipitation_rate', 'radar_reflectivity', 'liquid_water_content']].to_dataframe().describe()